# Toolbox With Skill — `@azure/ai-projects`

This notebook deploys a code-based **Hosted Agent** that discovers and uses **skills** from a Foundry Toolbox MCP endpoint. It:

1. Creates a shipping-cost **skill**.
2. Creates a **toolbox** version that references the skill (with a `toolbox_search_preview` tool).
3. Uploads the `toolbox-agent` code zip as a new Hosted Agent version, forwarding the project endpoint, model name, and toolbox MCP URL.
4. Assigns the hosted agent identity the **Foundry User** role so it can reach the toolbox MCP endpoint.
5. Waits for the version to become active and routes the agent endpoint to it.
6. Sends a query via the Responses API, approving any pending MCP skill-load requests.
7. Restores the previous endpoint and cleans up created resources (agent version, toolbox, and skill).

The hosted agent must already exist; create it first with the [`createHostedAgentFromImage`](./createHostedAgentFromImage.ipynb) sample.

It mirrors the [`toolboxWithSkill.ts`](./toolboxWithSkill.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` (and the `az` CLI used by the RBAC cell) can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`, `FOUNDRY_HOSTED_AGENT_NAME` (optional; defaults to `MyHostedAgent`), `AZURE_SUBSCRIPTION_ID` (optional; falls back to the active `az` subscription).

**Note on RBAC:** the toolbox requires the hosted agent's instance identity to have the **Foundry User** role on the Foundry account so it can call the toolbox MCP endpoint. Without it, the skill never loads. The RBAC cell assigns this role via the `az` CLI. Set `SKIP_RBAC=true` if the assignment is managed out-of-band.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  AgentEndpointConfig,
  CreateAgentVersionFromCodeContent,
  HostedAgentDefinition,
  ToolboxSearchPreviewToolboxTool,
  ToolboxSkillReference,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { createHash } from "node:crypto";
import { readFileSync } from "node:fs";
import path from "node:path";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const modelName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
const agentName = process.env["FOUNDRY_HOSTED_AGENT_NAME"] ?? "MyHostedAgent";

const codeZipPath = path.resolve("../assets/toolbox-agent.zip");
const skillName = "shipping-cost-skill";
const toolboxName = "toolbox_with_skill";

function sha256Hex(data: Uint8Array): string {
  return createHash("sha256").update(data).digest("hex");
}

console.log(`Model: ${modelName}`);
console.log(`Agent: ${agentName}`);

Model: gpt-5.2
Agent: MyHostedAgent9
Agent: MyHostedAgent9


In [2]:
// Create the AI Project client
// Annotated as `any` so tslab does not try to emit non-portable declarations
// referencing the deep `node_modules/openai` (pnpm junction) path.
const project: any = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Delete any pre-existing toolbox and skill with the same names (ignore 404),
// then create a shipping-cost skill.
const deleteExisting = async () => {
  try {
    await project.toolboxes.delete(toolboxName);
  } catch {
    // ignore 404
  }
  try {
    await project.beta.skills.delete(skillName);
  } catch {
    // ignore 404
  }
};
await deleteExisting();

const skillVersion = await project.beta.skills.create(skillName, {
  inlineContent: {
    description: "Compute shipping cost for a package given weight and destination.",
    instructions:
      "You are a shipping cost calculator. When asked to compute " +
      "shipping cost, use this formula: cost (USD) = 5 + 2 * weight_kg " +
      "for domestic destinations, and cost (USD) = 15 + 4 * weight_kg " +
      "for international destinations. Always state the formula you used.",
    metadata: { revision: "1" },
  },
});
console.log(`Created skill: ${skillVersion.name} version=${skillVersion.version}`);

Created skill: shipping-cost-skill version=1


In [4]:
// Create a toolbox that references the skill
const toolboxVersion = await project.toolboxes.createVersion(
  toolboxName,
  [{ type: "toolbox_search_preview" } as ToolboxSearchPreviewToolboxTool],
  {
    description: "Toolbox exposing a shipping-cost skill.",
    skills: [
      {
        type: "skill_reference",
        name: skillVersion.name,
        version: skillVersion.version,
      } as ToolboxSkillReference,
    ],
  },
);
console.log(`Created toolbox: ${toolboxVersion.name} version=${toolboxVersion.version}`);

Created toolbox: toolbox_with_skill version=1


In [ ]:
// Create a hosted agent version from the toolbox agent code
const toolboxMcpUrl = `${projectEndpoint}/toolboxes/${toolboxName}/versions/${toolboxVersion.version}/mcp?api-version=v1`;

const codeZip = readFileSync(codeZipPath);
const codeZipSha256 = sha256Hex(codeZip);

const definition: HostedAgentDefinition = {
  kind: "hosted",
  cpu: "0.5",
  memory: "1Gi",
  protocol_versions: [{ protocol: "responses", version: "2.0.0" }],
  code_configuration: {
    runtime: "python_3_14",
    entry_point: ["python", "main.py"],
    dependency_resolution: "remote_build",
  },
  environment_variables: {
    FOUNDRY_PROJECT_ENDPOINT: projectEndpoint,
    FOUNDRY_MODEL_NAME: modelName,
    MCP_SERVER_URL: toolboxMcpUrl,
  },
};

const content: CreateAgentVersionFromCodeContent = {
  metadata: {
    description: "Hosted agent code for toolbox MCP skills with shipping-cost skill.",
    definition,
  },
  code: { contents: codeZip, contentType: "application/zip", filename: "code.zip" },
};

console.log("Creating code-based hosted agent version...");
const created = await project.agents.createVersionFromCode(agentName, codeZipSha256, content);
const createdVersion = created.version;
console.log(`Created code-based hosted agent version: ${createdVersion}`);

// Capture the existing endpoint so it can be restored during cleanup.
const originalAgentEndpoint = (await project.agents.get(agentName)).agent_endpoint;

In [7]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, createdVersion);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: creating (attempt 1/60)
Agent version status: creating (attempt 2/60)
Agent version status: active (attempt 3/60)


In [8]:
// Route the agent endpoint to the new version
const endpointConfig: AgentEndpointConfig = {
  version_selector: {
    version_selection_rules: [
      {
        type: "FixedRatio",
        agent_version: createdVersion,
        traffic_percentage: 100,
      },
    ],
  },
  protocol_configuration: { responses: {} },
};
await project.agents.updateAgent(agentName, { agentEndpoint: endpointConfig });
console.log(`Agent endpoint configured for version ${createdVersion}`);

Agent endpoint configured for version 124


In [10]:
// Invoke the agent via the OpenAI Responses API
// Annotated as `any` so tslab does not emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient({
  azureConfig: { allowPreview: true, agentName: agentName },
});

const userInput = "Compute the shipping cost for a 3 kg package shipped domestically.";
console.log(`User: ${userInput}`);
let response = await openAIClient.responses.create({ input: userInput });

// Loading a toolbox skill goes through an MCP tool call that requires
// approval. Approve any pending requests and resubmit until the agent
// produces its final answer.
for (let round = 0; round < 5; round++) {
  const approvals: any[] = [];
  for (const item of response.output) {
    if (item.type === "mcp_approval_request" && item.id) {
      console.log(`Approving MCP request: ${item.name} (id: ${item.id})`);
      approvals.push({
        type: "mcp_approval_response",
        approval_request_id: item.id,
        approve: true,
      });
    }
  }
  if (approvals.length === 0) break;
  response = await openAIClient.responses.create({
    input: approvals,
    previous_response_id: response.id,
  });
}

console.log("Response:");
console.log(response.output_text);

User: Compute the shipping cost for a 3 kg package shipped domestically.
Response:
Domestic shipping cost formula: **cost (USD) = 5 + 2 × weight_kg**.

For **3 kg**:  
**cost = 5 + 2 × 3 = 11 USD**.


In [ ]:
// Clean up created resources. Restore the previous endpoint before deleting the
// temporary version so the agent endpoint is never left pointing at a deleted
// resource.
if (originalAgentEndpoint !== undefined) {
  await project.agents.updateAgent(agentName, { agentEndpoint: originalAgentEndpoint });
  console.log("Agent endpoint restored to previous configuration");
}
await project.agents.deleteVersion(agentName, createdVersion, { force: true });
console.log(`Agent version ${createdVersion} deleted.`);
await project.toolboxes.delete(toolboxName);
console.log("Toolbox deleted");
await project.beta.skills.delete(skillName);
console.log("Skill deleted");